# 🎵 KKBox Churn Prediction — Modeling v1
## Logistic Regression · LightGBM · XGBoost · Ensemble

---

## 📌 Mục tiêu

Xây dựng và so sánh các mô hình dự đoán churn cho KKBox, căn chỉnh theo
**Bryan Gregory's winning solution** (WSDM KKBox 2018).

---

## 🏆 Bryan Gregory's approach

> *"I used LightGBM and XGBoost in a weighted ensemble: 12% LGBM + 88% XGB"*
> — Bryan Gregory, WSDM KKBox 2018 Winner

| Mô hình | Tỷ lệ | Vai trò |
|---------|-------|---------|
| Logistic Regression | baseline | Sanity check, tuyến tính |
| LightGBM | 12% | Nhanh, tốt với categorical |
| XGBoost | 88% | Chính xác hơn, Bryan's primary |
| **Ensemble** | **100%** | **Kết hợp theo tỷ lệ Bryan** |

---

## 📅 Data pipeline

```
master_model_table.parquet
  Train : Jan 2017 — 992,931 users — churn 6.39%  (official labels)
  Val   : Feb 2017 — 970,960 users — churn 8.99%  (official labels)

inference_snapshot.parquet
  Inf   : Mar 2017 — 907,471 users — no labels (submission target)
```

---

## 🔑 Metric: AUC (Area Under ROC Curve)

Competition đánh giá bằng AUC. Gini = 2×AUC - 1 dùng để so sánh.

---

## 📋 Checklist

- [x] Official labels (train.csv / train_v2.csv)
- [x] 27 features, no leakage
- [x] Time-based split (train < val < inf)
- [ ] Logistic Regression baseline
- [ ] LightGBM + Optuna 50+ trials
- [ ] XGBoost + Optuna 50+ trials
- [ ] Ensemble LGBM 12% + XGB 88%
- [ ] Submission CSV


In [5]:
# ===== 1. Imports =====
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.pipeline import Pipeline

# Tree models
import lightgbm as lgb
import xgboost as xgb

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)
print("✅ Imports OK")
print(f"  lightgbm : {lgb.__version__}")
print(f"  xgboost  : {xgb.__version__}")
print(f"  optuna   : {optuna.__version__}")


✅ Imports OK
  lightgbm : 4.6.0
  xgboost  : 3.0.5
  optuna   : 4.6.0


In [6]:
# ===== 2. Paths + Load =====
DATA_DIR    = Path("Data")
MODELS_DIR  = Path("Models")
MODELS_DIR.mkdir(exist_ok=True)

MASTER_PATH    = DATA_DIR / "master_model_table.parquet"
INFERENCE_PATH = DATA_DIR / "inference_snapshot.parquet"
FE_META_PATH   = DATA_DIR / "feature_engineering_metadata_v7.json"

for p in [MASTER_PATH, INFERENCE_PATH]:
    status = "FOUND ✅" if p.exists() else "MISSING ❌"
    print(f"{p}: {status}")

master = pd.read_parquet(MASTER_PATH)
inf_df = pd.read_parquet(INFERENCE_PATH)

fe_meta = {}
if FE_META_PATH.exists():
    with open(FE_META_PATH) as f:
        fe_meta = json.load(f)

print(f"\nmaster shape    : {master.shape}")
print(f"inference shape : {inf_df.shape}")
print(f"\nColumns         : {list(master.columns)}")
print(f"\nRows by split:")
print(master.groupby("dataset_split").size())
print(f"\nChurn by split:")
print(master.groupby("dataset_split")["is_churn"].mean())


Data\master_model_table.parquet: FOUND ✅
Data\inference_snapshot.parquet: FOUND ✅

master shape    : (1963891, 33)
inference shape : (32808, 29)

Columns         : ['msno', 'snapshot_date', 'last_expire', 'is_churn', 'label_source', 'dataset_split', 'bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'total_amount_paid', 'avg_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'payment_method_nunique', 'days_last_txn_to_expire']

Rows by split:
dataset_split
train         992931
validation    970960
dtype: int64

Churn by split:
dataset_split
train        0.063923
validation   0.089942
Name: is_churn, dtype: float64


In [7]:
# ===== 3. Data Preparation =====

# ── Meta + feature columns ────────────────────────────────────────────────────
META_COLS = ["msno", "snapshot_date", "last_expire",
             "is_churn", "label_source", "dataset_split"]
FEATURE_COLS = [c for c in master.columns if c not in META_COLS]
TARGET = "is_churn"

CAT_COLS = ["city", "gender", "registered_via"]  # need encoding
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

print(f"Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nCategorical  ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numeric      ({len(NUM_COLS)}): {NUM_COLS}")

# ── Train / Val split ─────────────────────────────────────────────────────────
train_df = master[master["dataset_split"] == "train"].copy()
val_df   = master[master["dataset_split"] == "validation"].copy()

X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET].astype(int)
X_val   = val_df[FEATURE_COLS].copy()
y_val   = val_df[TARGET].astype(int)
X_inf   = inf_df[FEATURE_COLS].copy()

print(f"\nX_train : {X_train.shape}  |  y_train churn: {y_train.mean():.4f}")
print(f"X_val   : {X_val.shape}  |  y_val   churn: {y_val.mean():.4f}")
print(f"X_inf   : {X_inf.shape}")

# ── NaN verification (FE pipeline guarantees zero NaN/inf in parquet) ──────
nan_train = X_train.isna().sum().sum()
nan_val   = X_val.isna().sum().sum()
nan_inf   = X_inf.isna().sum().sum()
print(f"\nNaN check — X_train: {nan_train}, X_val: {nan_val}, X_inf: {nan_inf}")
if nan_train > 0 or nan_val > 0:
    print("  ⚠️  NaN detected — re-run FE notebook to regenerate clean parquet")
else:
    print("  ✅ No NaN — ready for modeling")

# ── Ordinal encode categoricals (for LGBM, XGB, and LR) ──────────────────────
# Convert StringDtype NA → plain str 'Unknown' before OrdinalEncoder
# OrdinalEncoder cannot handle pandas NAType mixed with str
for df in [X_train, X_val, X_inf]:
    for c in CAT_COLS:
        df[c] = df[c].astype(object).fillna("Unknown").astype(str)

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
oe.fit(X_train[CAT_COLS])

for df in [X_train, X_val, X_inf]:
    df[CAT_COLS] = oe.transform(df[CAT_COLS]).astype(float)

print("\nOrdinal encoding done ✅")
print(f"  city categories     : {len(oe.categories_[0])}")
print(f"  gender categories   : {len(oe.categories_[1])}")
print(f"  registered_via cats : {len(oe.categories_[2])}")

# ── Class imbalance ratio (for XGB scale_pos_weight) ─────────────────────────
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
SCALE_POS_WEIGHT = n_neg / n_pos
print(f"\nClass imbalance — neg: {n_neg:,}  pos: {n_pos:,}  ratio: {SCALE_POS_WEIGHT:.2f}")


Feature cols (27): ['bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'total_amount_paid', 'avg_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'payment_method_nunique', 'days_last_txn_to_expire']

Categorical  (3): ['city', 'gender', 'registered_via']
Numeric      (24): ['bd', 'bd_missing', 'city_missing', 'gender_missing', 'registered_via_missing', 'days_since_reg', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'total_amount_paid', 'avg_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'payment_method_nunique', 'days_last_txn_to_expire']

X_train

## Bước 4 — Logistic Regression (Baseline)

Mô hình tuyến tính đơn giản nhất. Mục đích:
- Xác nhận features có signal (AUC > 0.5)
- Là lower bound để so sánh với LGBM/XGB
- Nếu LR đã tốt → features mạnh, bài toán không phức tạp về mặt tuyến tính

**Thiết lập:**
- `class_weight='balanced'` — xử lý imbalance
- `StandardScaler` — LR nhạy cảm với scale của features
- `max_iter=1000` — đủ để hội tụ
- `C=1.0` — regularization mặc định (L2)


In [9]:
print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_val  :", X_val.isna().sum().sum())

display(
    X_train.isna().mean().sort_values(ascending=False).head(20)
)

NaN in X_train: 2019360
NaN in X_val  : 1865064


bd                       0.084739
bd_missing               0.084739
city_missing             0.084739
n_txns                   0.084739
gender_missing           0.084739
days_since_reg           0.084739
registered_via_missing   0.084739
auto_renew_rate          0.084739
plan_days_std            0.084739
avg_plan_days            0.084739
last_is_auto_renew       0.084739
share_30d                0.084739
share_90d                0.084739
days_since_last_txn      0.084739
plan_change_flag         0.084739
total_amount_paid        0.084739
avg_amount_paid          0.084739
zero_paid_rate           0.084739
avg_discount_rate        0.084739
cancel_rate              0.084739
dtype: float64

In [10]:
# ===== 4. Logistic Regression Baseline =====
import time
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

start_lr = time.time()

# Keep only numeric columns for this simple LR baseline
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()

for col in X_train_lr.columns:
    X_train_lr[col] = pd.to_numeric(X_train_lr[col], errors="coerce")
    X_val_lr[col] = pd.to_numeric(X_val_lr[col], errors="coerce")

# Impute missing values first
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train_lr)
X_val_imputed   = imputer.transform(X_val_lr)

# Scale for LR only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled   = scaler.transform(X_val_imputed)

lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    C=1.0,
    solver="saga",   # good for large datasets
    random_state=42,
    n_jobs=-1,
)

lr.fit(X_train_scaled, y_train)

lr_val_proba = lr.predict_proba(X_val_scaled)[:, 1]
lr_auc       = roc_auc_score(y_val, lr_val_proba)
lr_logloss   = log_loss(y_val, lr_val_proba)
lr_gini      = 2 * lr_auc - 1
lr_time      = time.time() - start_lr

print("=" * 55)
print("LOGISTIC REGRESSION — RESULTS")
print("=" * 55)
print(f"  Val AUC      : {lr_auc:.6f}")
print(f"  Val Gini     : {lr_gini:.6f}")
print(f"  Val LogLoss  : {lr_logloss:.6f}")
print(f"  Train time   : {lr_time:.1f}s")
print("=" * 55)

LR_RESULTS = {
    "model": "Logistic Regression",
    "auc": lr_auc,
    "gini": lr_gini,
    "logloss": lr_logloss,
    "time": lr_time
}

LOGISTIC REGRESSION — RESULTS
  Val AUC      : 0.683501
  Val Gini     : 0.367003
  Val LogLoss  : 0.613538
  Train time   : 126.3s


## Bước 5 — LightGBM + Optuna Tuning

LightGBM là mô hình gradient boosting tối ưu cho dữ liệu tabular lớn.
Trong Bryan's ensemble, LGBM đóng góp **12%** của prediction cuối.

**Optuna tuning:**
- 50 trials tối thiểu
- Objective: maximize val AUC
- Search space bao gồm: `num_leaves`, `learning_rate`, `min_child_samples`,
  `feature_fraction`, `bagging_fraction`, `lambda_l1`, `lambda_l2`, `max_depth`

**Sau khi tìm best params:**
- Retrain với full train set và best params
- Evaluate trên val set
- Lưu model để dùng trong ensemble


In [11]:
# ===== 5. LightGBM + Optuna =====
N_TRIALS_LGBM = 50

def lgbm_objective(trial):
    params = {
        "objective"          : "binary",
        "metric"             : "auc",
        "verbosity"          : -1,
        "boosting_type"      : "gbdt",
        "random_state"       : 42,
        "n_jobs"             : -1,
        "is_unbalance"       : True,
        "num_leaves"         : trial.suggest_int("num_leaves", 20, 300),
        "learning_rate"      : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "n_estimators"       : trial.suggest_int("n_estimators", 100, 1000),
        "min_child_samples"  : trial.suggest_int("min_child_samples", 10, 200),
        "feature_fraction"   : trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction"   : trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq"       : 1,
        "lambda_l1"          : trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2"          : trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "max_depth"          : trial.suggest_int("max_depth", 3, 12),
        "min_split_gain"     : trial.suggest_float("min_split_gain", 0.0, 1.0),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

print(f"Starting LGBM Optuna ({N_TRIALS_LGBM} trials)...")
start_lgbm_tune = time.time()

lgbm_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

lgbm_tune_time = time.time() - start_lgbm_tune
print(f"\nLGBM tuning done in {lgbm_tune_time:.0f}s")
print(f"Best trial AUC: {lgbm_study.best_value:.6f}")
print(f"Best params   : {lgbm_study.best_params}")

# ── Retrain with best params ──────────────────────────────────────────────────
best_lgbm_params = {
    "objective": "binary", "metric": "auc", "verbosity": -1,
    "boosting_type": "gbdt", "random_state": 42, "n_jobs": -1,
    "is_unbalance": True, **lgbm_study.best_params
}

start_lgbm_fit = time.time()
lgbm_model = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

lgbm_val_proba = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_auc       = roc_auc_score(y_val, lgbm_val_proba)
lgbm_logloss   = log_loss(y_val, lgbm_val_proba)
lgbm_gini      = 2 * lgbm_auc - 1
lgbm_time      = time.time() - start_lgbm_fit

print("\n" + "=" * 55)
print("LIGHTGBM — FINAL RESULTS")
print("=" * 55)
print(f"  Val AUC      : {lgbm_auc:.6f}")
print(f"  Val Gini     : {lgbm_gini:.6f}")
print(f"  Val LogLoss  : {lgbm_logloss:.6f}")
print(f"  Best iter    : {lgbm_model.best_iteration_}")
print(f"  Fit time     : {lgbm_time:.1f}s")
print("=" * 55)

LGBM_RESULTS = {"model": "LightGBM", "auc": lgbm_auc,
                "gini": lgbm_gini, "logloss": lgbm_logloss,
                "time": lgbm_tune_time + lgbm_time}


Starting LGBM Optuna (50 trials)...


Best trial: 43. Best value: 0.770119: 100%|██████████| 50/50 [02:01<00:00,  2.43s/it]



LGBM tuning done in 122s
Best trial AUC: 0.770119
Best params   : {'num_leaves': 37, 'learning_rate': 0.06522936158642315, 'n_estimators': 105, 'min_child_samples': 50, 'feature_fraction': 0.8091654091817151, 'bagging_fraction': 0.6289940905488517, 'lambda_l1': 0.004362643927128067, 'lambda_l2': 1.5848141923589252, 'max_depth': 5, 'min_split_gain': 0.7342490210753466}

LIGHTGBM — FINAL RESULTS
  Val AUC      : 0.767163
  Val Gini     : 0.534327
  Val LogLoss  : 0.390129
  Best iter    : 15
  Fit time     : 2.2s


## Bước 6 — XGBoost + Optuna Tuning

XGBoost là **mô hình chính** trong Bryan's ensemble (**88%** của prediction cuối).

**So sánh với LGBM:**
- XGBoost thường chính xác hơn nhưng chậm hơn
- LightGBM tốt hơn với dữ liệu thưa và categorical
- Kết hợp cả hai giúp capture các patterns khác nhau

**`scale_pos_weight`** = `n_negative / n_positive` — xử lý class imbalance cho XGB.

**Optuna tuning:**
- 50 trials, objective: maximize val AUC
- `eval_metric='auc'`, `early_stopping_rounds=30`


In [12]:
# ===== 6. XGBoost + Optuna =====
N_TRIALS_XGB = 50

def xgb_objective(trial):
    params = {
        "objective"          : "binary:logistic",
        "eval_metric"        : "auc",
        "random_state"       : 42,
        "n_jobs"             : -1,
        "verbosity"          : 0,
        "scale_pos_weight"   : SCALE_POS_WEIGHT,
        "n_estimators"       : trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate"      : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth"          : trial.suggest_int("max_depth", 3, 10),
        "min_child_weight"   : trial.suggest_int("min_child_weight", 1, 50),
        "subsample"          : trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree"   : trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel"  : trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "reg_alpha"          : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda"         : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma"              : trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=30,
                               enable_categorical=False)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

print(f"Starting XGB Optuna ({N_TRIALS_XGB} trials)...")
start_xgb_tune = time.time()

xgb_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

xgb_tune_time = time.time() - start_xgb_tune
print(f"\nXGB tuning done in {xgb_tune_time:.0f}s")
print(f"Best trial AUC: {xgb_study.best_value:.6f}")
print(f"Best params   : {xgb_study.best_params}")

# ── Retrain with best params ──────────────────────────────────────────────────
best_xgb_params = {
    "objective": "binary:logistic", "eval_metric": "auc",
    "random_state": 42, "n_jobs": -1, "verbosity": 0,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "early_stopping_rounds": 50,
    "enable_categorical": False,
    **xgb_study.best_params
}

start_xgb_fit = time.time()
xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=False)

xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_auc       = roc_auc_score(y_val, xgb_val_proba)
xgb_logloss   = log_loss(y_val, xgb_val_proba)
xgb_gini      = 2 * xgb_auc - 1
xgb_time      = time.time() - start_xgb_fit

print("\n" + "=" * 55)
print("XGBOOST — FINAL RESULTS")
print("=" * 55)
print(f"  Val AUC      : {xgb_auc:.6f}")
print(f"  Val Gini     : {xgb_gini:.6f}")
print(f"  Val LogLoss  : {xgb_logloss:.6f}")
print(f"  Best iter    : {xgb_model.best_iteration}")
print(f"  Fit time     : {xgb_time:.1f}s")
print("=" * 55)

XGB_RESULTS = {"model": "XGBoost", "auc": xgb_auc,
               "gini": xgb_gini, "logloss": xgb_logloss,
               "time": xgb_tune_time + xgb_time}


Starting XGB Optuna (50 trials)...


Best trial: 40. Best value: 0.770878: 100%|██████████| 50/50 [08:53<00:00, 10.66s/it]



XGB tuning done in 533s
Best trial AUC: 0.770878
Best params   : {'n_estimators': 293, 'learning_rate': 0.17097133513262772, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.9549021014853643, 'colsample_bytree': 0.4039461435679026, 'colsample_bylevel': 0.586384271456029, 'reg_alpha': 9.267759098134674e-05, 'reg_lambda': 2.245235804727654e-05, 'gamma': 2.094304184867286}

XGBOOST — FINAL RESULTS
  Val AUC      : 0.770878
  Val Gini     : 0.541755
  Val LogLoss  : 0.556714
  Best iter    : 17
  Fit time     : 8.8s


## Bước 7 — So sánh các mô hình

So sánh AUC, Gini, và LogLoss trên validation set (Feb 2017).

- **AUC**: càng cao càng tốt (1.0 = perfect, 0.5 = random)
- **Gini = 2×AUC - 1**: thường dùng trong ngành tài chính/subscription
- **LogLoss**: càng thấp càng tốt (đo calibration của probability)


In [13]:
# ===== 7. Model Comparison =====
results = pd.DataFrame([LR_RESULTS, LGBM_RESULTS, XGB_RESULTS])
results = results.sort_values("auc", ascending=False).reset_index(drop=True)
results["rank"] = results.index + 1

print("=" * 65)
print("MODEL COMPARISON — Validation Set (Feb 2017)")
print("=" * 65)
print(results[["rank","model","auc","gini","logloss","time"]].to_string(index=False))
print("=" * 65)
print(f"\nBest model: {results.iloc[0]['model']} (AUC = {results.iloc[0]['auc']:.6f})")
print(f"LR baseline AUC : {LR_RESULTS['auc']:.6f}")
print(f"LGBM gain vs LR : +{LGBM_RESULTS['auc'] - LR_RESULTS['auc']:.6f}")
print(f"XGB  gain vs LR : +{XGB_RESULTS['auc']  - LR_RESULTS['auc']:.6f}")

display(results[["model","auc","gini","logloss"]].style.background_gradient(
    subset=["auc","gini"], cmap="Greens"
).background_gradient(subset=["logloss"], cmap="Reds_r").format({
    "auc": "{:.6f}", "gini": "{:.6f}", "logloss": "{:.6f}"
}))


MODEL COMPARISON — Validation Set (Feb 2017)
 rank               model      auc     gini  logloss       time
    1             XGBoost 0.770878 0.541755 0.556714 541.847351
    2            LightGBM 0.767163 0.534327 0.390129 123.847227
    3 Logistic Regression 0.683501 0.367003 0.613538 126.279101

Best model: XGBoost (AUC = 0.770878)
LR baseline AUC : 0.683501
LGBM gain vs LR : +0.083662
XGB  gain vs LR : +0.087376


,model,auc,gini,logloss
0,XGBoost,0.770878,0.541755,0.556714
1,LightGBM,0.767163,0.534327,0.390129
2,Logistic Regression,0.683501,0.367003,0.613538


## Bước 8 — Ensemble: LGBM 12% + XGB 88% (Bryan's Setup)

### Tại sao ensemble?

Mỗi mô hình capture các patterns khác nhau:
- **LightGBM**: tốt hơn với features thưa, categorical, noise
- **XGBoost**: tốt hơn với numeric dense features, tổng quát hóa tốt hơn

Kết hợp theo tỷ lệ Bryan giúp giảm variance và tăng AUC.

### Công thức

```python
ensemble_proba = 0.12 × lgbm_proba + 0.88 × xgb_proba
```

### Thử nghiệm thêm

Ngoài tỷ lệ Bryan (12/88), ta cũng thử grid search tìm tỷ lệ tối ưu.


In [14]:
# ===== 8. Ensemble =====

# ── Bryan's 12/88 blend ───────────────────────────────────────────────────────
LGBM_WEIGHT = 0.12
XGB_WEIGHT  = 0.88

ensemble_val_proba = LGBM_WEIGHT * lgbm_val_proba + XGB_WEIGHT * xgb_val_proba
ensemble_auc       = roc_auc_score(y_val, ensemble_val_proba)
ensemble_logloss   = log_loss(y_val, ensemble_val_proba)
ensemble_gini      = 2 * ensemble_auc - 1

print("=" * 55)
print(f"ENSEMBLE (LGBM {LGBM_WEIGHT*100:.0f}% + XGB {XGB_WEIGHT*100:.0f}%) — BRYAN'S SETUP")
print("=" * 55)
print(f"  Val AUC      : {ensemble_auc:.6f}")
print(f"  Val Gini     : {ensemble_gini:.6f}")
print(f"  Val LogLoss  : {ensemble_logloss:.6f}")
print(f"  vs LGBM alone: {ensemble_auc - lgbm_auc:+.6f}")
print(f"  vs XGB  alone: {ensemble_auc - xgb_auc:+.6f}")
print("=" * 55)

# ── Grid search best blend ratio ──────────────────────────────────────────────
print("\nSearching optimal blend ratio...")
best_w, best_blend_auc = LGBM_WEIGHT, ensemble_auc

for w_lgbm in np.arange(0.0, 1.01, 0.05):
    w_xgb  = 1.0 - w_lgbm
    proba  = w_lgbm * lgbm_val_proba + w_xgb * xgb_val_proba
    auc    = roc_auc_score(y_val, proba)
    if auc > best_blend_auc:
        best_blend_auc = auc
        best_w         = w_lgbm

print(f"  Optimal LGBM weight : {best_w:.2f}  (XGB: {1-best_w:.2f})")
print(f"  Optimal blend AUC   : {best_blend_auc:.6f}")
print(f"  Bryan's blend AUC   : {ensemble_auc:.6f}")
print(f"  Difference          : {best_blend_auc - ensemble_auc:+.6f}")

# ── Use best blend for final ensemble ────────────────────────────────────────
FINAL_LGBM_W = best_w
FINAL_XGB_W  = 1.0 - best_w
final_ensemble_val_proba = FINAL_LGBM_W * lgbm_val_proba + FINAL_XGB_W * xgb_val_proba
final_ensemble_auc       = roc_auc_score(y_val, final_ensemble_val_proba)

print(f"\nFinal ensemble weights: LGBM={FINAL_LGBM_W:.2f}, XGB={FINAL_XGB_W:.2f}")
print(f"Final ensemble AUC    : {final_ensemble_auc:.6f}")

ENSEMBLE_RESULTS = {
    "model": f"Ensemble (LGBM {FINAL_LGBM_W:.0%} + XGB {FINAL_XGB_W:.0%})",
    "auc": final_ensemble_auc,
    "gini": 2*final_ensemble_auc - 1,
    "logloss": log_loss(y_val, final_ensemble_val_proba),
    "time": 0,
}


ENSEMBLE (LGBM 12% + XGB 88%) — BRYAN'S SETUP
  Val AUC      : 0.771434
  Val Gini     : 0.542867
  Val LogLoss  : 0.532158
  vs LGBM alone: +0.004270
  vs XGB  alone: +0.000556

Searching optimal blend ratio...
  Optimal LGBM weight : 0.45  (XGB: 0.55)
  Optimal blend AUC   : 0.772321
  Bryan's blend AUC   : 0.771434
  Difference          : +0.000887

Final ensemble weights: LGBM=0.45, XGB=0.55
Final ensemble AUC    : 0.772321


## Bước 9 — Feature Importance

So sánh feature importance giữa LGBM và XGB.
Features quan trọng nhất thường là:
- `days_since_last_txn` — recency
- `days_last_txn_to_expire` — khoảng cách giao dịch cuối đến hết hạn
- `auto_renew_rate` / `last_is_auto_renew` — ý định tự động gia hạn
- `cancel_rate` / `cancel_count` — lịch sử hủy
- `tenure_days` — tuổi tài khoản


In [15]:
# ===== 9. Feature Importance =====

# LGBM importance
lgbm_imp = pd.DataFrame({
    "feature"    : FEATURE_COLS,
    "lgbm_gain"  : lgbm_model.booster_.feature_importance(importance_type="gain"),
    "lgbm_split" : lgbm_model.booster_.feature_importance(importance_type="split"),
}).sort_values("lgbm_gain", ascending=False).reset_index(drop=True)

# XGB importance
xgb_imp_raw = xgb_model.get_booster().get_score(importance_type="gain")
xgb_imp = pd.DataFrame([
    {"feature": f, "xgb_gain": xgb_imp_raw.get(f, 0)} for f in FEATURE_COLS
]).sort_values("xgb_gain", ascending=False).reset_index(drop=True)

# Merge for comparison
imp_combined = lgbm_imp.merge(xgb_imp, on="feature", how="outer").fillna(0)
imp_combined["lgbm_gain_norm"] = (imp_combined["lgbm_gain"] /
                                   imp_combined["lgbm_gain"].sum() * 100)
imp_combined["xgb_gain_norm"]  = (imp_combined["xgb_gain"] /
                                   imp_combined["xgb_gain"].sum() * 100)
imp_combined = imp_combined.sort_values("lgbm_gain_norm", ascending=False)

print("=" * 65)
print("FEATURE IMPORTANCE (Gain %) — Top 15")
print("=" * 65)
top15 = imp_combined.head(15)[["feature","lgbm_gain_norm","xgb_gain_norm"]]
top15.columns = ["Feature", "LGBM Gain%", "XGB Gain%"]
print(top15.to_string(index=False))
print("=" * 65)

display(
    imp_combined.head(15)[["feature","lgbm_gain_norm","xgb_gain_norm"]]
    .rename(columns={"feature":"Feature","lgbm_gain_norm":"LGBM Gain%","xgb_gain_norm":"XGB Gain%"})
    .style.background_gradient(subset=["LGBM Gain%","XGB Gain%"], cmap="Blues")
    .format({"LGBM Gain%":"{:.2f}","XGB Gain%":"{:.2f}"})
    .hide(axis="index")
)


FEATURE IMPORTANCE (Gain %) — Top 15
                Feature  LGBM Gain%  XGB Gain%
         registered_via   43.274534  26.790762
days_last_txn_to_expire   17.935540   5.260355
         days_since_reg   12.260749   3.553498
        auto_renew_rate    8.808272   4.590177
     last_is_auto_renew    6.483201   9.737281
                     bd    3.110428   2.139593
                   city    2.606619   2.534487
      total_amount_paid    1.162145   2.186696
        avg_amount_paid    0.828923   2.772844
    days_since_last_txn    0.738222   1.038372
              share_30d    0.596716   0.817486
                 n_txns    0.503841   5.942786
             bd_missing    0.467437  15.333631
                 gender    0.441756   0.501473
            cancel_rate    0.285359   1.699212


Feature,LGBM Gain%,XGB Gain%
registered_via,43.27,26.79
days_last_txn_to_expire,17.94,5.26
days_since_reg,12.26,3.55
auto_renew_rate,8.81,4.59
last_is_auto_renew,6.48,9.74
bd,3.11,2.14
city,2.61,2.53
total_amount_paid,1.16,2.19
avg_amount_paid,0.83,2.77
days_since_last_txn,0.74,1.04


## Bước 10 — Inference + Submission

Dùng **final ensemble** để predict trên March 2017 population (907,471 users).

```
ensemble_proba = LGBM_WEIGHT × lgbm_proba + XGB_WEIGHT × xgb_proba
```

Output: `submission.csv` với format của competition:
```
msno, is_churn
user_1, 0.123
user_2, 0.456
...
```

`is_churn` là **xác suất** (probability), không phải nhãn 0/1.


In [16]:
# ===== 10. Inference — March 2017 =====
print(f"Generating predictions for {X_inf.shape[0]:,} users...")
start_inf = time.time()

lgbm_inf_proba = lgbm_model.predict_proba(X_inf)[:, 1]
xgb_inf_proba  = xgb_model.predict_proba(X_inf)[:, 1]
ensemble_inf_proba = FINAL_LGBM_W * lgbm_inf_proba + FINAL_XGB_W * xgb_inf_proba

inf_time = time.time() - start_inf

print(f"  Inference time: {inf_time:.1f}s")
print(f"  Prediction stats:")
print(f"    Mean  : {ensemble_inf_proba.mean():.4f}")
print(f"    Median: {np.median(ensemble_inf_proba):.4f}")
print(f"    Std   : {ensemble_inf_proba.std():.4f}")
print(f"    Min   : {ensemble_inf_proba.min():.4f}")
print(f"    Max   : {ensemble_inf_proba.max():.4f}")
print(f"    % > 0.5: {(ensemble_inf_proba > 0.5).mean():.2%}")


Generating predictions for 32,808 users...
  Inference time: 0.0s
  Prediction stats:
    Mean  : 0.4766
    Median: 0.4832
    Std   : 0.1357
    Min   : 0.1685
    Max   : 0.8395
    % > 0.5: 46.37%


In [17]:
# ===== 11. Export Submission + Model Artifacts =====
import pickle

# ── Submission CSV ─────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    "msno"    : inf_df["msno"].values,
    "is_churn": ensemble_inf_proba,
})
submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"✅ Submission saved: {submission_path}")
print(f"   Shape   : {submission.shape}")
print(f"   Preview :")
display(submission.head())

# ── Also save individual model predictions ────────────────────────────────────
lgbm_sub = pd.DataFrame({"msno": inf_df["msno"].values, "is_churn": lgbm_inf_proba})
xgb_sub  = pd.DataFrame({"msno": inf_df["msno"].values, "is_churn": xgb_inf_proba})
lgbm_sub.to_csv(DATA_DIR / "submission_lgbm.csv", index=False)
xgb_sub.to_csv(DATA_DIR / "submission_xgb.csv",   index=False)
print(f"✅ LGBM-only submission : Data/submission_lgbm.csv")
print(f"✅ XGB-only submission  : Data/submission_xgb.csv")

# ── Save models ────────────────────────────────────────────────────────────────
lgbm_model.booster_.save_model(str(MODELS_DIR / "lgbm_model.txt"))
xgb_model.save_model(str(MODELS_DIR  / "xgb_model.json"))
with open(MODELS_DIR / "ordinal_encoder.pkl", "wb") as f:
    pickle.dump(oe, f)

print(f"\n✅ Models saved to Models/")
print(f"   lgbm_model.txt")
print(f"   xgb_model.json")
print(f"   ordinal_encoder.pkl")

# ── Save modeling metadata ─────────────────────────────────────────────────────
model_meta = {
    "feature_cols"       : FEATURE_COLS,
    "cat_cols"           : CAT_COLS,
    "num_cols"           : NUM_COLS,
    "target"             : TARGET,
    "ensemble_lgbm_weight": FINAL_LGBM_W,
    "ensemble_xgb_weight" : FINAL_XGB_W,
    "val_results": {
        "logistic_regression": LR_RESULTS,
        "lightgbm"           : LGBM_RESULTS,
        "xgboost"            : XGB_RESULTS,
        "ensemble"           : ENSEMBLE_RESULTS,
    },
    "lgbm_best_params" : lgbm_study.best_params,
    "xgb_best_params"  : xgb_study.best_params,
    "n_trials_lgbm"    : N_TRIALS_LGBM,
    "n_trials_xgb"     : N_TRIALS_XGB,
}
with open(DATA_DIR / "modeling_metadata.json", "w") as f:
    json.dump(model_meta, f, indent=2, default=str)
print(f"✅ Metadata saved: Data/modeling_metadata.json")


✅ Submission saved: Data\submission.csv
   Shape   : (32808, 2)
   Preview :


,msno,is_churn
0,QzyX6ufzLhmGeiE0cyytk0lC13hQp45q4BOVUL+zpTI=,0.486616
1,fkLgfIOX0bWM9/BQQChOCDzoos23szsckxPvxrBbtmY=,0.602049
2,etZ6WY4Qrn76v3JqLj0xpT1zA3yCoL35lti7cLc4tik=,0.689357
3,nq+4KRKNWTQkH9VNArdNfhBNl70Vh01WEi/i9rPlxqU=,0.358150
4,c8MWafMse+6+aWZwpBddB5CQ6dH2uncxqSJX/hoj8f0=,0.506282


✅ LGBM-only submission : Data/submission_lgbm.csv
✅ XGB-only submission  : Data/submission_xgb.csv

✅ Models saved to Models/
   lgbm_model.txt
   xgb_model.json
   ordinal_encoder.pkl
✅ Metadata saved: Data/modeling_metadata.json


In [18]:
# ===== 12. Final Summary =====
all_results = pd.DataFrame([LR_RESULTS, LGBM_RESULTS, XGB_RESULTS, ENSEMBLE_RESULTS])
all_results = all_results.sort_values("auc", ascending=False).reset_index(drop=True)

print("=" * 65)
print("FINAL LEADERBOARD — Validation AUC (Feb 2017)")
print("=" * 65)
for _, row in all_results.iterrows():
    bar = "█" * int(row["auc"] * 50)
    print(f"  {row['model']:<40} AUC: {row['auc']:.6f}")
print("=" * 65)
print(f"\nBest model   : {all_results.iloc[0]['model']}")
print(f"Best Val AUC : {all_results.iloc[0]['auc']:.6f}")
print(f"Best Val Gini: {all_results.iloc[0]['gini']:.6f}")
print(f"\nEnsemble vs LGBM alone : {ENSEMBLE_RESULTS['auc'] - LGBM_RESULTS['auc']:+.6f}")
print(f"Ensemble vs XGB  alone : {ENSEMBLE_RESULTS['auc'] - XGB_RESULTS['auc']:+.6f}")
print(f"\nSubmission files:")
print(f"  Data/submission.csv       ← Main (ensemble)")
print(f"  Data/submission_lgbm.csv  ← LGBM only")
print(f"  Data/submission_xgb.csv   ← XGB only")
print(f"\nEnsemble weights used: LGBM={FINAL_LGBM_W:.0%}, XGB={FINAL_XGB_W:.0%}")
print(f"Bryan's setup          : LGBM=12%, XGB=88%")

display(all_results[["model","auc","gini","logloss"]].style
    .background_gradient(subset=["auc","gini"], cmap="Greens")
    .background_gradient(subset=["logloss"], cmap="Reds_r")
    .format({"auc":"{:.6f}","gini":"{:.6f}","logloss":"{:.6f}"})
    .hide(axis="index"))


FINAL LEADERBOARD — Validation AUC (Feb 2017)
  Ensemble (LGBM 45% + XGB 55%)            AUC: 0.772321
  XGBoost                                  AUC: 0.770878
  LightGBM                                 AUC: 0.767163
  Logistic Regression                      AUC: 0.683501

Best model   : Ensemble (LGBM 45% + XGB 55%)
Best Val AUC : 0.772321
Best Val Gini: 0.544641

Ensemble vs LGBM alone : +0.005157
Ensemble vs XGB  alone : +0.001443

Submission files:
  Data/submission.csv       ← Main (ensemble)
  Data/submission_lgbm.csv  ← LGBM only
  Data/submission_xgb.csv   ← XGB only

Ensemble weights used: LGBM=45%, XGB=55%
Bryan's setup          : LGBM=12%, XGB=88%


model,auc,gini,logloss
Ensemble (LGBM 45% + XGB 55%),0.772321,0.544641,0.471868
XGBoost,0.770878,0.541755,0.556714
LightGBM,0.767163,0.534327,0.390129
Logistic Regression,0.683501,0.367003,0.613538
